In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
path = r'D:/Universidad/5to Semestre/Inteligencia Artificial/git/git/Datasets Preparados/D3 NHANES (National Health and Nutrition Examination Survey)/nhanes_full_prepared.csv '
df = pd.read_csv(path)

# Ahora X tiene muchas más columnas (una por cada categoría encontrada)
X = df.values.astype(np.float32)
y = df['LBDGLUSI'].values.astype(np.float32)

In [ ]:
X_train = X[:80000]
y_train = y[:80000]

X_test = X[80000:90000]
y_test = y[80000:90000]

In [ ]:
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

y_mean = y_train.mean()
y_std = y_train.std()

# normalizar
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

y_train = (y_train - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

In [ ]:
class DatasetRegresion(torch.utils.data.Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1,1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = DatasetRegresion(X_train, y_train)
test_dataset = DatasetRegresion(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
n_entradas = X.shape[1]

model = torch.nn.Sequential(
    torch.nn.Linear(n_entradas, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 1)
)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# --- CONFIGURACIÓN PREVIA ---
epochs = 1000
best_loss = float("inf")

history_loss = []
history_train_mae = [] # Cambiamos Accuracy por MAE (Mean Absolute Error)
history_test_mae = []

for epoch in range(epochs):
    # --- FASE 1: ENTRENAMIENTO ---
    model.train()
    epoch_loss_list = []
    train_mae_total = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculamos el error en dólares (desnormalizando si es necesario)
        # Aquí calculamos el error absoluto simple entre la predicción y el real
        error_absoluto = torch.abs(y_pred - y_batch).sum().item()
        train_mae_total += error_absoluto
        train_total += y_batch.size(0)

        epoch_loss_list.append(loss.item())

    # --- FASE 2: PRUEBA ---
    model.eval()
    test_mae_total = 0
    test_total = 0

    with torch.no_grad():
        for X_test_batch, y_test_batch in test_loader:
            y_test_pred = model(X_test_batch)
            error_test = torch.abs(y_test_pred - y_test_batch).sum().item()
            test_mae_total += error_test
            test_total += y_test_batch.size(0)

    # --- MÉTRICAS ---
    avg_loss = np.mean(epoch_loss_list)
    mae_train = train_mae_total / train_total
    mae_test = test_mae_total / test_total

    history_loss.append(avg_loss)
    history_train_mae.append(mae_train)
    history_test_mae.append(mae_test)

    # Checkpoint
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "mejor_modelo_salarios.pt")

    if (epoch + 1) % 100 == 0:
        print(f"📊 [Ep {epoch+1}] Loss: {avg_loss:.4f}")

In [ ]:
model.load_state_dict(torch.load("mejor_modelo_salarios.pt"))
model.eval()

In [ ]:
model.eval()
n_puntos = 100

# 1. Tomar 100 datos del set de prueba
X_sample = torch.tensor(X_test[:n_puntos], dtype=torch.float32)
y_real = y_test[:n_puntos] * y_std + y_mean # Desnormalizar

# 2. Predicción
with torch.no_grad():
    y_pred_norm = model(X_sample).numpy().flatten()
    y_pred = y_pred_norm * y_std + y_mean # Desnormalizar

# 3. Graficar
plt.figure(figsize=(8, 6))
plt.scatter(y_real, y_pred, color='blue', alpha=0.6, label='Predicciones')
plt.plot([y_real.min(), y_real.max()], [y_real.min(), y_real.max()], 'r--', lw=2, label='Línea Ideal (Perfecta)')

plt.title('Comparación: Salario Real vs. Predicción (100 casos)')
plt.xlabel('Salario Real (USD)')
plt.ylabel('Predicción del Modelo (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_loss, label='Pérdida de Entrenamiento')
plt.title('Evolución de la Pérdida (MSE)')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# --- EVALUACIÓN FINAL ---
model.eval()

with torch.no_grad():
    # Predicciones entrenamiento
    y_train_pred = model(torch.tensor(X_train, dtype=torch.float32)).numpy()
    y_test_pred = model(torch.tensor(X_test, dtype=torch.float32)).numpy()

# Desnormalizar (MUY IMPORTANTE)
y_train_real = y_train * y_std + y_mean
y_test_real = y_test * y_std + y_mean

y_train_pred = y_train_pred * y_std + y_mean
y_test_pred = y_test_pred * y_std + y_mean

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# MAE
mae_train_final = mean_absolute_error(y_train_real, y_train_pred)
mae_test_final = mean_absolute_error(y_test_real, y_test_pred)

# R2
r2_train = r2_score(y_train_real, y_train_pred)
r2_test = r2_score(y_test_real, y_test_pred)

In [ ]:
# Promedio de los valores reales (en dólares)
mean_salary = y_train_real.mean()

# Precisión tipo porcentaje
precision_train = (1 - (mae_train_final / mean_salary)) * 100
precision_test = (1 - (mae_test_final / mean_salary)) * 100

# Limitar entre 0 y 100
precision_train = max(0, min(100, precision_train))
precision_test = max(0, min(100, precision_test))

In [ ]:
tabla_precision = pd.DataFrame({
    "Tipo de Datos": ["Entrenamiento", "Prueba"],
    "Precisión (%)": [precision_train, precision_test]
})

print(tabla_precision)

In [ ]:
plt.figure()
plt.bar(tabla_precision["Tipo de Datos"], tabla_precision["Precisión (%)"])
plt.title("Precisión del modelo (Entrenamiento vs Prueba)")
plt.ylabel("Precisión (%)")
plt.xlabel("Tipo de Datos")
plt.ylim(0, 100)
plt.show()

In [ ]:
model.eval()
y_preds = []
y_reales = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        y_pred = model(X_batch).squeeze()
        y_preds.extend(y_pred.numpy())
        y_reales.extend(y_batch.numpy())

y_preds = np.vstack(y_preds)
y_reales = np.vstack(y_reales)

In [ ]:
preds_real = np.array(y_preds) * y_std + y_mean
reales_real = np.array(y_reales) * y_std + y_mean

r2 = r2_score(reales_real, preds_real)
print("R2:", r2)

In [ ]:
model.eval()
n_ejemplos = 5

# 1. Tomamos los datos de prueba (que están normalizados)
X_test_tensor = torch.tensor(X_test[:n_ejemplos], dtype=torch.float32)

# 2. Predicción del modelo
with torch.no_grad():
    predicciones_norm = model(X_test_tensor).numpy().flatten()

# 3. RECUPERAR LOS VALORES REALES (Desnormalización)
# Desnormalizamos tanto la realidad como la predicción para que la comparación sea justa
y_reales_usd = y_test[:n_ejemplos] * y_std + y_mean
y_preds_usd = predicciones_norm * y_std + y_mean

# 4. Imprimir la tabla corregida
print(f"{'CASO':<5} | {'SALARIO REAL':<15} | {'PREDICCIÓN':<15} | {'ERROR %'}")
print("-" * 65)

for i in range(n_ejemplos):
    real = y_reales_usd[i]
    pred = y_preds_usd[i]

    # Calculamos el error porcentual: (Diferencia / Real) * 100
    error_pct = (abs(real - pred) / real) * 100

    print(f"{i+1:<5} | ${real:>12,.2f} | ${pred:>12,.2f} | {error_pct:>8.2f}%")